# POC Model Walkthrough

This notebook compares two versions of the same binary classifier:

- `with_2022_t1`: uses 2017 T1, 2017 T2, 2022 T1, and security features
- `without_2022_t1`: removes all 2022 T1 election features

Target:

- `1` if Macron wins the commune in 2022 second round
- `0` otherwise

In [121]:
from __future__ import annotations

import csv
import io
import json
import math
import random
from pathlib import Path

ROOT_DIR = Path.cwd().resolve().parents[1] if Path.cwd().name == 'ml' else Path.cwd().resolve()
while not (ROOT_DIR / 'data').exists() and ROOT_DIR != ROOT_DIR.parent:
    ROOT_DIR = ROOT_DIR.parent

ELECTION_DIR = ROOT_DIR / 'data' / 'gold' / 'election'
SECURITY_DIR = ROOT_DIR / 'data' / 'gold' / 'security'
OUTPUT_DIR = ROOT_DIR / 'src' / 'ml' / 'outputs'

print('ROOT_DIR =', ROOT_DIR)
print('ELECTION_DIR =', ELECTION_DIR)
print('SECURITY_DIR =', SECURITY_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)

ROOT_DIR = /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc
ELECTION_DIR = /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/data/gold/election
SECURITY_DIR = /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/data/gold/security
OUTPUT_DIR = /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/src/ml/outputs


In [122]:
def read_csv_rows(path: Path) -> list[dict[str, str]]:
    content = path.read_text(encoding='utf-8')
    return list(csv.DictReader(io.StringIO(content), delimiter=';'))


def write_csv_rows(path: Path, rows: list[dict[str, object]], fieldnames: list[str]) -> None:
    with path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def normalize_id_commune(value: str) -> str:
    parts = str(value).split('-', 1)
    if len(parts) == 2:
        dept = parts[0].strip().zfill(2)
        commune = parts[1].strip().replace('.0', '').zfill(3)
        return f'{dept}-{commune}'
    return str(value).strip()


def slugify(text: str) -> str:
    text = text.lower().strip()
    for old, new in [
        ('é', 'e'), ('è', 'e'), ('ê', 'e'), ('ë', 'e'),
        ('à', 'a'), ('â', 'a'), ('ä', 'a'),
        ('î', 'i'), ('ï', 'i'), ('ô', 'o'), ('ö', 'o'),
        ('ù', 'u'), ('û', 'u'), ('ü', 'u'), ('ç', 'c'),
        ("'", '_'), ('-', '_'), (' ', '_'), ('(', '_'), (')', '_'), ('/', '_')
    ]:
        text = text.replace(old, new)
    while '__' in text:
        text = text.replace('__', '_')
    return text.strip('_')


def to_float(value: str, default: float = 0.0) -> float:
    if value is None:
        return default
    raw = str(value).strip()
    if raw == '':
        return default
    try:
        return float(raw)
    except ValueError:
        return default


def to_int(value: str, default: int = 0) -> int:
    if value is None:
        return default
    raw = str(value).strip()
    if raw == '':
        return default
    try:
        return int(float(raw))
    except ValueError:
        return default


def sigmoid(x: float) -> float:
    x = max(-50.0, min(50.0, x))
    return 1.0 / (1.0 + math.exp(-x))


def dot_product(a: list[float], b: list[float]) -> float:
    return sum(x * y for x, y in zip(a, b))


def mean(values: list[float]) -> float:
    return sum(values) / len(values) if values else 0.0


def std(values: list[float], avg: float) -> float:
    if not values:
        return 1.0
    variance = sum((v - avg) ** 2 for v in values) / len(values)
    return math.sqrt(variance) or 1.0


print('Basic helper functions loaded.')

Basic helper functions loaded.


In [123]:
def classification_metrics(y_true: list[int], y_pred: list[int]) -> dict[str, float]:
    total = len(y_true)
    correct = sum(1 for yt, yp in zip(y_true, y_pred) if yt == yp)
    accuracy = correct / total if total else 0.0

    tp = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 1 and yp == 1)
    tn = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 0 and yp == 0)
    fp = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 0 and yp == 1)
    fn = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 1 and yp == 0)

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    balanced_accuracy = (recall + specificity) / 2.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    return {
        'accuracy': accuracy,
        'balanced_accuracy': balanced_accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'tp': tp,
        'tn': tn,
        'fp': fp,
        'fn': fn,
    }


def stratified_split(rows: list[dict[str, object]], target_key: str, test_size: float = 0.2, seed: int = 42):
    random.seed(seed)
    grouped = {0: [], 1: []}
    for row in rows:
        grouped[int(row[target_key])].append(row)

    train_rows = []
    test_rows = []

    for _, group in grouped.items():
        random.shuffle(group)
        split_idx = max(1, int(round(len(group) * (1 - test_size))))
        train_rows.extend(group[:split_idx])
        test_rows.extend(group[split_idx:])

    random.shuffle(train_rows)
    random.shuffle(test_rows)
    return train_rows, test_rows


def prepare_matrices(rows: list[dict[str, object]], feature_names: list[str]):
    matrix = []
    for row in rows:
        matrix.append([to_float(row[feature], default=float('nan')) for feature in feature_names])
    return matrix


def impute_and_scale(train_matrix: list[list[float]], test_matrix: list[list[float]]):
    n_features = len(train_matrix[0])
    means = []
    stds = []

    for j in range(n_features):
        values = [row[j] for row in train_matrix if not math.isnan(row[j])]
        avg = mean(values)
        s = std(values, avg)
        means.append(avg)
        stds.append(s)

    def transform(matrix: list[list[float]]) -> list[list[float]]:
        transformed = []
        for row in matrix:
            new_row = []
            for j, value in enumerate(row):
                v = means[j] if math.isnan(value) else value
                new_row.append((v - means[j]) / stds[j])
            transformed.append(new_row)
        return transformed

    return transform(train_matrix), transform(test_matrix)


def fit_logistic_regression(X: list[list[float]], y: list[int], lr: float = 0.05, epochs: int = 3000, l2: float = 0.001):
    n_features = len(X[0])
    weights = [0.0] * n_features
    bias = 0.0
    n_samples = len(X)

    for epoch in range(epochs):
        grad_w = [0.0] * n_features
        grad_b = 0.0

        for row, target in zip(X, y):
            pred = sigmoid(dot_product(row, weights) + bias)
            error = pred - target
            for j in range(n_features):
                grad_w[j] += error * row[j]
            grad_b += error

        for j in range(n_features):
            grad_w[j] = grad_w[j] / n_samples + l2 * weights[j]
            weights[j] -= lr * grad_w[j]
        bias -= lr * (grad_b / n_samples)

        if epoch in [0, 499, 999, 1999, 2999]:
            print(f'Epoch {epoch + 1}/{epochs} finished')

    return weights, bias


def predict_classes(X: list[list[float]], weights: list[float], bias: float):
    probas = [sigmoid(dot_product(row, weights) + bias) for row in X]
    preds = [1 if p >= 0.5 else 0 for p in probas]
    return preds, probas


print('Training helper functions loaded.')

Training helper functions loaded.


In [124]:
dim_election = read_csv_rows(ELECTION_DIR / 'dim_election.csv')
dim_candidat = read_csv_rows(ELECTION_DIR / 'dim_candidat.csv')
fact_resultats = read_csv_rows(ELECTION_DIR / 'fact_resultats.csv')
dim_nuance = read_csv_rows(ELECTION_DIR / 'dim_nuance.csv')
dim_indicateur = read_csv_rows(SECURITY_DIR / 'dim_indicateur_securite.csv')
fact_securite = read_csv_rows(SECURITY_DIR / 'fact_securite.csv')

print('dim_election rows =', len(dim_election))
print('dim_candidat rows =', len(dim_candidat))
print('fact_resultats rows =', len(fact_resultats))
print('dim_nuance rows =', len(dim_nuance))
print('dim_indicateur rows =', len(dim_indicateur))
print('fact_securite rows =', len(fact_securite))

print('Sample election row:', fact_resultats[0])
print('Sample security row:', fact_securite[0])

dim_election rows = 4
dim_candidat rows = 16
fact_resultats rows = 7378
dim_nuance rows = 86
dim_indicateur rows = 15
fact_securite rows = 37125
Sample election row: {'id_commune': '69-1', 'id_election': '2017_T2', 'id_candidat': 'CAND_001', 'inscrits': '257', 'abstentions': '39', 'votants': '218', 'blancs': '20', 'nuls': '7', 'exprimes': '191', 'pct_abs_ins': '15.18', 'pct_vot_ins': '84.82', 'pct_blancs_ins': '7.78', 'pct_blancs_vot': '9.17', 'pct_nuls_ins': '2.72', 'pct_nuls_vot': '3.21', 'pct_exprimes_ins': '74.32', 'pct_exprimes_vot': '87.61', 'voix': '93', 'pct_voix_ins': '36.19', 'pct_voix_exprimes': '48.69', 'date_integration_gold': '2026-03-26T12:04:56.534710'}
Sample security row: {'id_commune': '69-001', 'annee': '2016', 'id_indicateur_securite': 'SEC_001', 'code_departement': '69', 'code_commune': '001', 'code_insee_commune': '69001', 'nombre': '3.3925926', 'taux_pour_mille': '6.7806363', 'est_diffuse': 'ndiff', 'insee_pop': '358', 'insee_log': '163', 'complement_info_nombre

In [125]:
result_with_2022_t1 = run_experiment('with_2022_t1', include_2022_t1=True)
result_without_2022_t1 = run_experiment('without_2022_t1', include_2022_t1=False)           


EXPERIMENT = with_2022_t1
include_2022_t1 = True
Dataset rows = 267
Feature count = 111
Class balance = {0: 49, 1: 218}
Nuances features count = 22, examples: ['share_nuance_com_2022_t1', 'share_nuance_div_2017_t1', 'share_nuance_dlf_2017_t1', 'share_nuance_dlf_2022_t1', 'share_nuance_eelv_2022_t1']
Epoch 1/3000 finished
Epoch 500/3000 finished
Epoch 1000/3000 finished
Epoch 2000/3000 finished
Epoch 3000/3000 finished
Baseline metrics =
{
  "accuracy": 0.8148148148148148,
  "balanced_accuracy": 0.5,
  "precision": 0.8148148148148148,
  "recall": 1.0,
  "f1": 0.8979591836734693,
  "tp": 44,
  "tn": 0,
  "fp": 10,
  "fn": 0
}
Model metrics =
{
  "accuracy": 0.9259259259259259,
  "balanced_accuracy": 0.9159090909090909,
  "precision": 0.9761904761904762,
  "recall": 0.9318181818181818,
  "f1": 0.9534883720930233,
  "tp": 41,
  "tn": 9,
  "fp": 1,
  "fn": 3
}
Saved to = /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/src/ml/outputs/with_2022_t1

EXPERIMENT = without_2022_t1
in

In [126]:
comparison = {
    'with_2022_t1': {
        'features_count': result_with_2022_t1['features_count'],
        'accuracy': result_with_2022_t1['model_metrics']['accuracy'],
        'balanced_accuracy': result_with_2022_t1['model_metrics']['balanced_accuracy'],
        'f1': result_with_2022_t1['model_metrics']['f1'],
    },
    'without_2022_t1': {
        'features_count': result_without_2022_t1['features_count'],
        'accuracy': result_without_2022_t1['model_metrics']['accuracy'],
        'balanced_accuracy': result_without_2022_t1['model_metrics']['balanced_accuracy'],
        'f1': result_without_2022_t1['model_metrics']['f1'],
    },
}

print('COMPARISON SUMMARY')
print(json.dumps(comparison, indent=2))

(OUTPUT_DIR / 'comparison.json').write_text(json.dumps(comparison, indent=2), encoding='utf-8')
print('Comparison file saved to =', OUTPUT_DIR / 'comparison.json')

COMPARISON SUMMARY
{
  "with_2022_t1": {
    "features_count": 111,
    "accuracy": 0.9259259259259259,
    "balanced_accuracy": 0.9159090909090909,
    "f1": 0.9534883720930233
  },
  "without_2022_t1": {
    "features_count": 78,
    "accuracy": 0.9074074074074074,
    "balanced_accuracy": 0.865909090909091,
    "f1": 0.942528735632184
  }
}
Comparison file saved to = /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/src/ml/outputs/comparison.json


In [127]:
print('TOP 10 FEATURES WITH 2022 T1')
for row in result_with_2022_t1['feature_importance'][:10]:
    print(row)

print('\nTOP 10 FEATURES WITHOUT 2022 T1')
for row in result_without_2022_t1['feature_importance'][:10]:
    print(row)

TOP 10 FEATURES WITH 2022 T1
{'feature': 'share_candidat_macron_2022_t1', 'coefficient': 0.9563231964496245, 'abs_coefficient': 0.9563231964496245}
{'feature': 'share_nuance_rem_2022_t1', 'coefficient': 0.9563231964496245, 'abs_coefficient': 0.9563231964496245}
{'feature': 'share_candidat_le_pen_2022_t1', 'coefficient': -0.948610329729128, 'abs_coefficient': 0.948610329729128}
{'feature': 'share_nuance_rn_2022_t1', 'coefficient': -0.948610329729128, 'abs_coefficient': 0.948610329729128}
{'feature': 'pct_blancs_ins_2017_t2', 'coefficient': -0.77508745898734, 'abs_coefficient': 0.77508745898734}
{'feature': 'sec_rate_vols_avec_armes_2021', 'coefficient': 0.685507987413946, 'abs_coefficient': 0.685507987413946}
{'feature': 'sec_rate_violences_physiques_hors_cadre_familial_2022', 'coefficient': -0.5236768024446093, 'abs_coefficient': 0.5236768024446093}
{'feature': 'sec_rate_escroqueries_et_fraudes_aux_moyens_de_paiement_2022', 'coefficient': 0.4912456845697325, 'abs_coefficient': 0.491245

## Interpretation

- If the model without `2022 T1` still performs well, the POC is stronger because it is less dependent on the immediately previous round.
- The main file to cite in your report is `comparison.json`.
- The detailed outputs for each experiment are stored in:
  - `src/ml/outputs/with_2022_t1`
  - `src/ml/outputs/without_2022_t1`

## Nouvelle approche : Prédictions généralisées par Nuance Politique

L'objectif de cette cellule est de créer un **nouveau modèle** qui prédit la probabilité de victoire d'une **Nuance Politique spécifique** au second tour (ex: `REM` ou `RN`), en se basant *uniquement* sur les scores passés des nuances (et sans utiliser l'historique des noms des candidats).

Cela permet d'avoir un algorithme "agnostique" des personnes, qui pourrait s'appliquer théoriquement à d'autres contextes territoriaux en se basant uniquement sur la polarité politique.

In [128]:
def run_nuance_centric_model(target_nuance: str, experiment_name: str, include_2022_t1: bool = True):
    print('\n' + '=' * 80)
    print(f'NEW EXPERIMENT = {experiment_name} | TARGET NUANCE = {target_nuance}')
    print('=' * 80)

    # 1. Extraction mapping
    election_map = {row['id_election']: f"{row['annee_election']}_T{row['tour']}" for row in dim_election}
    candidate_map = {row['id_candidat']: {'nom': row['nom'], 'id_nuance': row.get('id_nuance', row.get('nuance', 'inconnu'))} for row in dim_candidat}

    feature_data = {}
    target_scores = {}

    # 2. Construction des features agnostiques des candidats
    for row in fact_resultats:
        id_commune = normalize_id_commune(row['id_commune'])
        election_key = election_map[row['id_election']]
        candidate = candidate_map[row['id_candidat']]
        commune_features = feature_data.setdefault(id_commune, {})

        if election_key == '2022_T2':
            # On cherche qui a gagné le T2 et sa NUANCE
            score = to_float(row['voix'])
            best = target_scores.get(id_commune)
            if best is None or score > best[0]:
                target_scores[id_commune] = (score, candidate['id_nuance'])
        else:
            if not include_2022_t1 and election_key == '2022_T1':
                continue
            for metric in ['inscrits', 'abstentions', 'votants', 'blancs', 'nuls', 'exprimes', 'pct_abs_ins', 'pct_vot_ins', 'pct_blancs_ins', 'pct_nuls_ins']:
                commune_features[f'{metric}_{election_key.lower()}'] = to_float(row[metric])
            
            pct_voix = to_float(row['pct_voix_exprimes'])
            
            # UNIQUE MODIFICATION: On ignore les noms des candidats, on se base exclusivement sur les nuances
            nuance_key = f"share_nuance_{slugify(candidate['id_nuance'])}_{election_key.lower()}"
            commune_features[nuance_key] = commune_features.get(nuance_key, 0.0) + pct_voix

    # 3. Y value = 1 si la nuance ciblée gagne
    targets = {id_commune: 1 if winner_nuance == target_nuance else 0 for id_commune, (_, winner_nuance) in target_scores.items()}
    security_features = build_security_dataset()

    # 4. Préparation Matrice
    all_feature_names = set()
    dataset_rows = []
    target_col = f'target_{target_nuance}_wins_2022_t2'

    for id_commune, target_y in targets.items():
        r = {'id_commune': id_commune, target_col: target_y}
        r.update(feature_data.get(id_commune, {}))
        r.update(security_features.get(id_commune, {}))
        dataset_rows.append(r)
        all_feature_names.update(k for k in r.keys() if k not in {'id_commune', target_col})

    feature_names = sorted(all_feature_names)
    for r in dataset_rows:
        for f in feature_names:
            r.setdefault(f, None)

    print('Dataset rows =', len(dataset_rows))
    print('Feature count =', len(feature_names))
    
    # Check what features look like
    print('Examples of features used:', [f for f in feature_names if 'share' in f][:5])

    train_rows, test_rows = stratified_split(dataset_rows, target_col, test_size=0.2, seed=42)
    train_matrix = prepare_matrices(train_rows, feature_names)
    test_matrix = prepare_matrices(test_rows, feature_names)
    train_matrix, test_matrix = impute_and_scale(train_matrix, test_matrix)

    y_train = [int(r[target_col]) for r in train_rows]
    y_test = [int(r[target_col]) for r in test_rows]

    weights, bias = fit_logistic_regression(train_matrix, y_train)
    pred_classes, pred_probas = predict_classes(test_matrix, weights, bias)

    model_metrics = classification_metrics(y_test, pred_classes)
    print('Model metrics =')
    print(json.dumps(model_metrics, indent=2))

    # 5. Export des résultats
    predictions = []
    for r, pred, proba in zip(test_rows, pred_classes, pred_probas):
        predictions.append({
            'id_commune': r['id_commune'],
            target_col: r[target_col],
            'predicted_class': pred,
            f'predicted_proba_{target_nuance}': proba,
        })

    feature_importance = []
    for feature, weight in sorted(zip(feature_names, weights), key=lambda item: abs(item[1]), reverse=True):
        feature_importance.append({
            'feature': feature,
            'coefficient': weight,
            'abs_coefficient': abs(weight),
        })

    experiment_dir = OUTPUT_DIR / experiment_name
    experiment_dir.mkdir(parents=True, exist_ok=True)

    write_csv_rows(experiment_dir / 'model_dataset.csv', dataset_rows, ['id_commune', target_col] + feature_names)
    write_csv_rows(experiment_dir / 'test_predictions.csv', predictions, ['id_commune', target_col, 'predicted_class', f'predicted_proba_{target_nuance}'])
    write_csv_rows(experiment_dir / 'feature_importance.csv', feature_importance, ['feature', 'coefficient', 'abs_coefficient'])

    print(f'Done! Predictions saved to: {experiment_dir / "test_predictions.csv"}')
    return feature_importance, predictions

print('Nuance model function loaded.')

Nuance model function loaded.


In [129]:
# Modèle ciblant la prédiction de la victoire de la nuance REM (majorité présidentielle)
feat_imp_rem, preds_rem = run_nuance_centric_model(
    target_nuance='REM', 
    experiment_name='nuance_predict_winner_REM', 
    include_2022_t1=True
)

print('\nTOP 10 VARIABLES Poussant à la victoire de la nuance REM :')
for row in feat_imp_rem[:10]:
    print(row)



NEW EXPERIMENT = nuance_predict_winner_REM | TARGET NUANCE = REM
Dataset rows = 267
Feature count = 86
Examples of features used: ['share_nuance_com_2022_t1', 'share_nuance_div_2017_t1', 'share_nuance_dlf_2017_t1', 'share_nuance_dlf_2022_t1', 'share_nuance_eelv_2022_t1']
Epoch 1/3000 finished
Epoch 500/3000 finished
Epoch 1000/3000 finished
Epoch 2000/3000 finished
Epoch 3000/3000 finished
Model metrics =
{
  "accuracy": 0.9259259259259259,
  "balanced_accuracy": 0.9159090909090909,
  "precision": 0.9761904761904762,
  "recall": 0.9318181818181818,
  "f1": 0.9534883720930233,
  "tp": 41,
  "tn": 9,
  "fp": 1,
  "fn": 3
}
Done! Predictions saved to: /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/src/ml/outputs/nuance_predict_winner_REM/test_predictions.csv

TOP 10 VARIABLES Poussant à la victoire de la nuance REM :
{'feature': 'share_nuance_rn_2022_t1', 'coefficient': -1.44102666465613, 'abs_coefficient': 1.44102666465613}
{'feature': 'share_nuance_rem_2022_t1', 'coefficie

## Modèle Multi-Classes (Prédiction de la nuance gagnante)

Plutôt que de faire une classification binaire (Vrai ou Faux pour une nuance spécifique), nous entraînons ici un modèle **Multi-Classes** avec l'approche "One-vs-Rest". L'algorithme calcule indépendamment la probabilité de victoire de chaque nuance, et prédit celle qui obtient le score de confiance le plus élevé. On a donc directement la nuance vainqueur pour chaque commune.

In [130]:
def run_multiclass_nuance_model(experiment_name: str, include_2022_t1: bool = True):
    print('\n' + '=' * 80)
    print(f'MULTICLASS EXPERIMENT = {experiment_name} | INCLUDE 2022 T1 = {include_2022_t1}')
    print('=' * 80)

    # 1. Extraction mapping
    election_map = {row['id_election']: f"{row['annee_election']}_T{row['tour']}" for row in dim_election}
    candidate_map = {row['id_candidat']: {'id_nuance': row.get('id_nuance', row.get('nuance', 'inconnu'))} for row in dim_candidat}

    feature_data = {}
    target_scores = {}

    # 2. Construction des features
    for row in fact_resultats:
        id_commune = normalize_id_commune(row['id_commune'])
        election_key = election_map[row['id_election']]
        candidate = candidate_map[row['id_candidat']]
        commune_features = feature_data.setdefault(id_commune, {})

        # On cible l'élection 2022 T2 pour déterminer la nuance gagnante finale
        if election_key == '2022_T2':
            score = to_float(row['voix'])
            best = target_scores.get(id_commune)
            if best is None or score > best[0]:
                target_scores[id_commune] = (score, candidate['id_nuance'])
        else:
            # STOP DATA LEAKAGE: Ignore 2022_T1 if requested
            if not include_2022_t1 and election_key == '2022_T1':
                continue
                
            pct_voix = to_float(row['pct_voix_exprimes'])
            nuance_key = f"share_nuance_{slugify(candidate['id_nuance'])}_{election_key.lower()}"
            commune_features[nuance_key] = commune_features.get(nuance_key, 0.0) + pct_voix
            
            for metric in ['inscrits', 'abstentions', 'votants', 'blancs', 'nuls', 'exprimes', 'pct_abs_ins', 'pct_vot_ins', 'pct_blancs_ins', 'pct_nuls_ins']:
                commune_features[f'{metric}_{election_key.lower()}'] = to_float(row[metric])

    # 3. Y value = La classe gagnante (ex: 'REM', 'RN')
    targets_nuance = {id_commune: nuance for id_commune, (_, nuance) in target_scores.items()}
    unique_classes = sorted(list(set(targets_nuance.values())))

    security_features = build_security_dataset()

    # 4. Préparation Matrice
    all_feature_names = set()
    dataset_rows = []

    for id_commune, actual_nuance in targets_nuance.items():
        r = {'id_commune': id_commune, 'target_nuance': actual_nuance}
        r.update(feature_data.get(id_commune, {}))
        r.update(security_features.get(id_commune, {}))
        dataset_rows.append(r)
        for k in r.keys():
             if k not in {'id_commune', 'target_nuance'}:
                 all_feature_names.add(k)

    feature_names = sorted(all_feature_names)
    for r in dataset_rows:
        for f in feature_names:
            if 'share_nuance' in f:
                r.setdefault(f, 0.0)
            else:
                r.setdefault(f, None)
                
    # Fix Split stratifié pour des keys en chaine de caractères
    random.seed(42)
    train_rows = []
    test_rows = []
    grouped = {}
    for r in dataset_rows:
        grouped.setdefault(r['target_nuance'], []).append(r)
    
    for cls, group in grouped.items():
        random.shuffle(group)
        split_idx = max(1, int(round(len(group) * 0.8))) 
        train_rows.extend(group[:split_idx])
        test_rows.extend(group[split_idx:])

    random.shuffle(train_rows)
    random.shuffle(test_rows)

    train_matrix = prepare_matrices(train_rows, feature_names)
    test_matrix = prepare_matrices(test_rows, feature_names)
    train_matrix, test_matrix = impute_and_scale(train_matrix, test_matrix)

    # 5. Entraînement "One-vs-Rest"
    models = {}
    for cls in unique_classes:
        y_train_binary = [1 if r['target_nuance'] == cls else 0 for r in train_rows]
        weights, bias = fit_logistic_regression(train_matrix, y_train_binary, epochs=1500)
        models[cls] = {'weights': weights, 'bias': bias}

    # 6. Prédiction
    predictions = []
    correct = 0
    
    for i, row in enumerate(test_rows):
        best_cls = None
        best_proba = -1
        
        for cls in unique_classes:
            w = models[cls]['weights']
            b = models[cls]['bias']
            p = sigmoid(dot_product(test_matrix[i], w) + b)
            if p > best_proba:
                best_proba = p
                best_cls = cls
                
        if best_cls == row['target_nuance']:
            correct += 1
            
    accuracy = correct / len(test_rows) if test_rows else 0
    print(f"\n✅ Multi-class Accuracy : {accuracy * 100:.2f} %")
    return predictions, accuracy

print('Fonction Multi-Classes réparée pour strings.')

Fonction Multi-Classes réparée pour strings.


In [131]:
# RUN 1: Avec les données du 1er tour de 2022 (Risque de temporal data leakage)
_, acc_with_t1 = run_multiclass_nuance_model('multiclass_with_2022_t1', include_2022_t1=True)

# RUN 2: Sans les données du 1er tour de 2022 (Prédiction sur le long terme)
_, acc_without_t1 = run_multiclass_nuance_model('multiclass_without_2022_t1', include_2022_t1=False)

print(f"\n--- CONCLUSION ---")
print(f"Accuracy avec le 1er tour juste avant l'élection: {acc_with_t1*100:.2f}%")
print(f"Accuracy sans le 1er tour (seulement 2017 + Sécurité): {acc_without_t1*100:.2f}%")


MULTICLASS EXPERIMENT = multiclass_with_2022_t1 | INCLUDE 2022 T1 = True
Epoch 1/1500 finished
Epoch 500/1500 finished
Epoch 1000/1500 finished
Epoch 1/1500 finished
Epoch 500/1500 finished
Epoch 1000/1500 finished

✅ Multi-class Accuracy : 92.59 %

MULTICLASS EXPERIMENT = multiclass_without_2022_t1 | INCLUDE 2022 T1 = False
Epoch 1/1500 finished
Epoch 500/1500 finished
Epoch 1000/1500 finished
Epoch 1/1500 finished
Epoch 500/1500 finished
Epoch 1000/1500 finished

✅ Multi-class Accuracy : 90.74 %

--- CONCLUSION ---
Accuracy avec le 1er tour juste avant l'élection: 92.59%
Accuracy sans le 1er tour (seulement 2017 + Sécurité): 90.74%


## Un modèle beaucoup plus "Intelligent" : Random Forest (Forêts Aléatoires)

La régression logistique linéaire que nous avons construite de zéro donne de bons résultats, mais elle ne peut pas comprendre les interactions complexes entre les variables (ex: effet combiné du taux d'abstention ET de l'insécurité).

Pour surpasser ses performances, nous utilisons ici **Scikit-Learn** et un **Random Forest Classifier**.  
- Il génère des centaines d'arbres de décision qui votent ensemble.
- Il est très résistant à l'Overfitting (grâce au paramètre `max_depth` et au *Bagging*).
- Il nous donne une "Feature Importance" beaucoup plus fidèle à la réalité car non impactée par la colinéarité des variables.

In [132]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

def run_smarter_rf_model(experiment_name: str, include_2022_t1: bool = True):
    print('\n' + '=' * 80)
    print(f'SMARTER MULTICLASS MODEL: RANDOM FOREST | INCLUDE 2022 T1 = {include_2022_t1}')
    print('=' * 80)

    # 1. Pipeline de récupération des données agnostiques (Nuances unqiuement)
    election_map = {row['id_election']: f"{row['annee_election']}_T{row['tour']}" for row in dim_election}
    candidate_map = {row['id_candidat']: {'id_nuance': row.get('id_nuance', row.get('nuance', 'inconnu'))} for row in dim_candidat}
    nuance_libelles = {row['id_nuance']: row['libelle_nuance'] for row in dim_nuance}

    feature_data = {}
    target_scores = {}

    for row in fact_resultats:
        id_commune = normalize_id_commune(row['id_commune'])
        election_key = election_map[row['id_election']]
        candidate = candidate_map[row['id_candidat']]
        commune_features = feature_data.setdefault(id_commune, {})

        if election_key == '2022_T2':
            score = to_float(row['voix'])
            best = target_scores.get(id_commune)
            # On veut prédire qui (quelle nuance) gagne le T2
            if best is None or score > best[0]:
                target_scores[id_commune] = (score, candidate['id_nuance'])
        else:
            if not include_2022_t1 and election_key == '2022_T1':
                continue
            pct_voix = to_float(row['pct_voix_exprimes'])
            nuance_key = f"share_nuance_{slugify(candidate['id_nuance'])}_{election_key.lower()}"
            commune_features[nuance_key] = commune_features.get(nuance_key, 0.0) + pct_voix
            
            for metric in ['inscrits', 'abstentions', 'votants', 'blancs', 'nuls', 'exprimes', 'pct_abs_ins', 'pct_vot_ins', 'pct_blancs_ins', 'pct_nuls_ins']:
                commune_features[f'{metric}_{election_key.lower()}'] = to_float(row[metric])

    targets_nuance = {id_commune: nuance for id_commune, (_, nuance) in target_scores.items()}
    security_features = build_security_dataset()

    # 2. Construction d'un Dataset tabulaire pour Pandas
    dataset_rows = []
    all_features = set()
    for id_commune, actual_nuance in targets_nuance.items():
        r = {'id_commune': id_commune, 'target_nuance': actual_nuance}
        r.update(feature_data.get(id_commune, {}))
        r.update(security_features.get(id_commune, {}))
        dataset_rows.append(r)
        for k in r.keys():
             if k not in {'id_commune', 'target_nuance'}:
                 all_features.add(k)

    df = pd.DataFrame(dataset_rows)
    feature_cols = sorted(list(all_features))
    
    # 3. Nettoyage intelligent des valeurs nulles
    for f in feature_cols:
        if 'share_nuance' in f:
            df[f] = df[f].fillna(0.0) # Un parti absent = 0 votes
        elif 'sec_rate_' in f:
            df[f] = df[f].fillna(0.0) # Pas de criminalité recensée = 0%
        else:
            df[f] = df[f].fillna(df[f].median()) # Statistiques manquantes (population) = Médiane locale

    print(f"Dataset shape : {df.shape}")

    # 4. Split Train / Test
    X = df[feature_cols]
    y = df['target_nuance']
    
    X_train, X_test, y_train, y_test, indices_train, indices_test = train_test_split(
        X, y, df['id_commune'], test_size=0.2, random_state=42, stratify=y
    )

    # 5. Entraînement du modèle (100 Arbres formés à profondeur maximale de 15)
    rf = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, class_weight='balanced')
    rf.fit(X_train, y_train)

    # 6. Evaluation du modèle
    preds = rf.predict(X_test)
    probas = rf.predict_proba(X_test)
    classes = rf.classes_
    acc = accuracy_score(y_test, preds)
    
    print(f"\n✅ Random Forest Test Accuracy : {acc * 100:.2f} %")
    print("\n--- RAPPORT PAR NUANCE (Classification Report) ---")
    noms_classes = [nuance_libelles.get(c, c) for c in classes]
    print(classification_report(y_test, preds, target_names=noms_classes, zero_division=0))

    # 7. Importance réelle des variables
    importances = rf.feature_importances_
    df_importances = pd.DataFrame({'Variable': feature_cols, 'Importance': importances})
    df_importances = df_importances.sort_values('Importance', ascending=False)
    
    print("\n🌲 TOP 10 DES VARIABLES LES PLUS DÉTECTÉES PAR LA FORÊT :")
    display(df_importances.head(10))
    
    # 8. Création du Dataframe de résultats (Test Set uniquement)
    results_test = pd.DataFrame({
        'id_commune': indices_test,
        'actual_winner': y_test,
        'actual_winner_label': [nuance_libelles.get(x, x) for x in y_test],
        'predicted_winner': preds,
        'predicted_winner_label': [nuance_libelles.get(x, x) for x in preds]
    })
    for i, c in enumerate(classes):
        results_test[f'proba_{c}'] = probas[:, i]

    # 8b. Création du Dataframe de résultats (Toutes les communes)
    all_preds = rf.predict(X)
    all_probas = rf.predict_proba(X)
    results_all = pd.DataFrame({
        'id_commune': df['id_commune'],
        'actual_winner': y,
        'actual_winner_label': [nuance_libelles.get(x, x) for x in y],
        'predicted_winner': all_preds,
        'predicted_winner_label': [nuance_libelles.get(x, x) for x in all_preds]
    })
    for i, c in enumerate(classes):
        results_all[f'proba_{c}'] = all_probas[:, i]
        
    # 9. Sauvegarde des résultats
    experiment_dir = OUTPUT_DIR / experiment_name
    experiment_dir.mkdir(parents=True, exist_ok=True)
    
    df.to_csv(experiment_dir / 'model_dataset.csv', index=False)
    results_test.to_csv(experiment_dir / 'test_predictions.csv', index=False)
    results_all.to_csv(experiment_dir / 'all_predictions.csv', index=False)
    df_importances.to_csv(experiment_dir / 'feature_importance.csv', index=False)
    
    print(f"\n📂 Résultats sauvegardés dans : {experiment_dir}")
        
    return rf, df_importances, results_all

# --- Exécution d'un "VRAI" test difficile (Sans les tendances du 1er tour de 2022) ---
rf_model, imp_df, all_predictions_df = run_smarter_rf_model(experiment_name='random_forest_without_t1', include_2022_t1=False)


SMARTER MULTICLASS MODEL: RANDOM FOREST | INCLUDE 2022 T1 = False
Dataset shape : (267, 67)

✅ Random Forest Test Accuracy : 92.59 %

--- RAPPORT PAR NUANCE (Classification Report) ---
                         precision    recall  f1-score   support

La République en marche       0.92      1.00      0.96        44
 Rassemblement National       1.00      0.60      0.75        10

               accuracy                           0.93        54
              macro avg       0.96      0.80      0.85        54
           weighted avg       0.93      0.93      0.92        54


🌲 TOP 10 DES VARIABLES LES PLUS DÉTECTÉES PAR LA FORÊT :


,Variable,Importance
58,share_nuance_rem_2017_t2,0.218058
61,share_nuance_rn_2017_t2,0.140040
60,share_nuance_rn_2017_t1,0.086424
57,share_nuance_rem_2017_t1,0.081026
62,share_nuance_soc_2017_t1,0.036439
53,share_nuance_dlf_2017_t1,0.029629
4,exprimes_2017_t1,0.019836
11,insee_pop_2022,0.017746
20,pct_vot_ins_2017_t1,0.016626
6,inscrits_2017_t1,0.016456



📂 Résultats sauvegardés dans : /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/src/ml/outputs/random_forest_without_t1


testing election 2022 t1 


## Test Ultime : Prédire le Premier Tour (2022 T1)

Le premier tour est beaucoup plus complexe car il y a une multitude de candidats et de nuances (LFI, LR, RN, REM, REC...). Ce modèle va tenter de deviner quelle nuance arrivera en tête dans chaque commune lors du **1er tour de 2022**, en s'appuyant *uniquement* sur les données historiques de 2017 et l'insécurité.

In [133]:
def run_rf_model_for_2022_t1(experiment_name: str):
    print('\n' + '=' * 80)
    print(f'PREDICTING 2022 ROUND 1 (T1) WINNER: RANDOM FOREST')
    print('=' * 80)

    # 1. Pipeline de récupération
    election_map = {row['id_election']: f"{row['annee_election']}_T{row['tour']}" for row in dim_election}
    candidate_map = {row['id_candidat']: {'id_nuance': row.get('id_nuance', row.get('nuance', 'inconnu'))} for row in dim_candidat}
    nuance_libelles = {row['id_nuance']: row['libelle_nuance'] for row in dim_nuance}

    feature_data = {}
    target_scores = {}

    for row in fact_resultats:
        id_commune = normalize_id_commune(row['id_commune'])
        election_key = election_map[row['id_election']]
        candidate = candidate_map[row['id_candidat']]
        commune_features = feature_data.setdefault(id_commune, {})

        # Cible = On cherche qui a gagné le T1 de 2022
        if election_key == '2022_T1':
            score = to_float(row['voix'])
            best = target_scores.get(id_commune)
            if best is None or score > best[0]:
                target_scores[id_commune] = (score, candidate['id_nuance'])
        
        # Features = Uniquement les données de 2017 (T1 et T2) pour éviter le leakage !
        elif election_key.startswith('2017'):
            pct_voix = to_float(row['pct_voix_exprimes'])
            nuance_key = f"share_nuance_{slugify(candidate['id_nuance'])}_{election_key.lower()}"
            commune_features[nuance_key] = commune_features.get(nuance_key, 0.0) + pct_voix
            
            for metric in ['inscrits', 'abstentions', 'votants', 'blancs', 'nuls', 'exprimes', 'pct_abs_ins', 'pct_vot_ins', 'pct_blancs_ins', 'pct_nuls_ins']:
                commune_features[f'{metric}_{election_key.lower()}'] = to_float(row[metric])

    targets_nuance = {id_commune: nuance for id_commune, (_, nuance) in target_scores.items()}
    security_features = build_security_dataset()

    # 2. Construction d'un Dataset tabulaire pour Pandas
    dataset_rows = []
    all_features = set()
    for id_commune, actual_nuance in targets_nuance.items():
        r = {'id_commune': id_commune, 'target_nuance': actual_nuance}
        r.update(feature_data.get(id_commune, {}))
        r.update(security_features.get(id_commune, {}))
        dataset_rows.append(r)
        for k in r.keys():
             if k not in {'id_commune', 'target_nuance'}:
                 all_features.add(k)

    df = pd.DataFrame(dataset_rows)
    feature_cols = sorted(list(all_features))
    
    # 3. Nettoyage intelligent des valeurs nulles
    for f in feature_cols:
        if 'share_nuance' in f:
            df[f] = df[f].fillna(0.0)
        elif 'sec_rate_' in f:
            df[f] = df[f].fillna(0.0)
        else:
            df[f] = df[f].fillna(df[f].median())

    # Suppression des classes ultra-minoritaires (1 seul représentant) 
    # car elles empêchent Sklearn de faire un "Stratify" propre lors de la division Train/Test
    class_counts = df['target_nuance'].value_counts()
    valid_classes = class_counts[class_counts > 1].index
    if len(valid_classes) < len(class_counts):
        print(f"⚠️ Retrait des nuances ayant gagné dans une seule commune (trop rare) : {set(class_counts.index) - set(valid_classes)}")
        df = df[df['target_nuance'].isin(valid_classes)]

    print(f"Dataset shape : {df.shape}")
    print(f"Nuances possibles à prédire : {df['target_nuance'].unique()}")

    # 4. Split Train / Test
    X = df[feature_cols]
    y = df['target_nuance']
    
    X_train, X_test, y_train, y_test, indices_train, indices_test = train_test_split(
        X, y, df['id_commune'], test_size=0.2, random_state=42, stratify=y
    )

    # 5. Entraînement du modèle
    rf = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, class_weight='balanced')
    rf.fit(X_train, y_train)

    # 6. Evaluation du modèle
    preds = rf.predict(X_test)
    probas = rf.predict_proba(X_test)
    classes = rf.classes_
    acc = accuracy_score(y_test, preds)
    
    print(f"\n✅ Random Forest Test Accuracy (2022 T1) : {acc * 100:.2f} %")
    print("\n--- RAPPORT PAR NUANCE (Classification Report) ---")
    noms_classes = [nuance_libelles.get(c, c) for c in classes]
    print(classification_report(y_test, preds, target_names=noms_classes, zero_division=0))

    # 7. Importance réelle des variables
    importances = rf.feature_importances_
    df_importances = pd.DataFrame({'Variable': feature_cols, 'Importance': importances})
    df_importances = df_importances.sort_values('Importance', ascending=False)
    
    print("\n🌲 TOP 10 DES VARIABLES LES PLUS DÉTECTÉES PAR LA FORÊT :")
    display(df_importances.head(10))
    
    # 8. Création des Dataframes de résultats
    results_test = pd.DataFrame({
        'id_commune': indices_test,
        'actual_winner': y_test,
        'actual_winner_label': [nuance_libelles.get(x, x) for x in y_test],
        'predicted_winner': preds,
        'predicted_winner_label': [nuance_libelles.get(x, x) for x in preds]
    })
    for i, c in enumerate(classes):
        results_test[f'proba_{c}'] = probas[:, i]

    all_preds = rf.predict(X)
    all_probas = rf.predict_proba(X)
    results_all = pd.DataFrame({
        'id_commune': df['id_commune'],
        'actual_winner': y,
        'actual_winner_label': [nuance_libelles.get(x, x) for x in y],
        'predicted_winner': all_preds,
        'predicted_winner_label': [nuance_libelles.get(x, x) for x in all_preds]
    })
    for i, c in enumerate(classes):
        results_all[f'proba_{c}'] = all_probas[:, i]
        
    # 9. Sauvegarde des résultats
    experiment_dir = OUTPUT_DIR / experiment_name
    experiment_dir.mkdir(parents=True, exist_ok=True)
    
    df.to_csv(experiment_dir / 'model_dataset.csv', index=False)
    results_test.to_csv(experiment_dir / 'test_predictions.csv', index=False)
    results_all.to_csv(experiment_dir / 'all_predictions.csv', index=False)
    df_importances.to_csv(experiment_dir / 'feature_importance.csv', index=False)
    
    print(f"\n📂 Résultats T1 sauvegardés dans : {experiment_dir}")
        
    return rf, df_importances, results_all

# --- Exécution pour le Premier Tour 2022 ---
rf_t1_model, imp_t1_df, all_t1_predictions_df = run_rf_model_for_2022_t1(experiment_name='random_forest_2022_t1')


PREDICTING 2022 ROUND 1 (T1) WINNER: RANDOM FOREST
Dataset shape : (267, 67)
Nuances possibles à prédire : ['RN' 'REM' 'FI']

✅ Random Forest Test Accuracy (2022 T1) : 88.89 %

--- RAPPORT PAR NUANCE (Classification Report) ---
                         precision    recall  f1-score   support

    La France insoumise       1.00      0.67      0.80         3
La République en marche       0.90      0.95      0.92        38
 Rassemblement National       0.83      0.77      0.80        13

               accuracy                           0.89        54
              macro avg       0.91      0.79      0.84        54
           weighted avg       0.89      0.89      0.89        54


🌲 TOP 10 DES VARIABLES LES PLUS DÉTECTÉES PAR LA FORÊT :


,Variable,Importance
58,share_nuance_rem_2017_t2,0.104043
61,share_nuance_rn_2017_t2,0.093039
0,abstentions_2017_t1,0.069310
2,blancs_2017_t1,0.062112
60,share_nuance_rn_2017_t1,0.055036
57,share_nuance_rem_2017_t1,0.038780
1,abstentions_2017_t2,0.036556
14,pct_abs_ins_2017_t1,0.036373
15,pct_abs_ins_2017_t2,0.033230
20,pct_vot_ins_2017_t1,0.030714



📂 Résultats T1 sauvegardés dans : /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/src/ml/outputs/random_forest_2022_t1


In [1]:
import pandas as pd
from IPython.display import display
pd.set_option('display.max_rows', 50)

print('\n' + '=' * 80)
print('🔎 VISUALISATION : QUELLE NUANCE VA GAGNER DANS CHAQUE COMMUNE ?')
print('=' * 80)

# 1. Charger les noms des communes depuis la couche Gold
df_communes = pd.read_csv(ELECTION_DIR / 'dim_commune.csv', sep=';')

# 2. Normaliser les IDs des communes pour que la jointure fonctionne ('69-1' -> '69-001')
df_communes['id_commune_norm'] = df_communes['id_commune'].apply(normalize_id_commune)

# 3. Fusionner les prédictions (ex: all_t1_predictions_df) avec le nom de la commune
df_display = pd.merge(all_t1_predictions_df, df_communes, left_on='id_commune', right_on='id_commune_norm', how='left', suffixes=('', '_drop'))

# Ajout du Score de Confiance (Probabilité associée au vainqueur prédit)
# On récupère dynamiquement la valeur dans la colonne "proba_XYZ" selon la victoire prédite
df_display['confidence_score'] = df_display.apply(lambda row: row[f"proba_{row['predicted_winner']}"], axis=1)

# 4. Sélectionner et renommer les colonnes pour une lecture facile
visu_df = df_display[['id_commune', 'libelle_commune', 'predicted_winner', 'predicted_winner_label', 'confidence_score']].copy()

visu_df = visu_df.rename(columns={
    'id_commune': 'Code Insee',
    'libelle_commune': 'Commune',
    'predicted_winner': 'Sigle Prédit',
    'predicted_winner_label': '🏆 Nuance Politique Gagnante Prédite (Libellé)',
    'confidence_score': 'Score de Confiance'
})

# Mise en forme du pourcentage
visu_df['Score de Confiance'] = (visu_df['Score de Confiance'] * 100).round(2).astype(str) + ' %'

# 5. Afficher le tableau complet dans l'interface
display(visu_df)


🔎 VISUALISATION : QUELLE NUANCE VA GAGNER DANS CHAQUE COMMUNE ?


NameError: name 'ELECTION_DIR' is not defined